In [ ]:
import random
import time
from datasets import load_dataset
from tqdm import tqdm
import re

# 튜터 노트: 안녕하세요! 🤗 오늘 우리가 탐험할 데이터셋은 고품질의 교육용 한국어 텍스트 코퍼스예요.
# 'eliceai/korean-fineweb-edu-demo' 데이터셋을 통해, 실제로 LLM이 어떻게 '지식을 구조화'하는지 시뮬레이션해 볼 거예요.

# 📚 데이터셋 정보
DATASET_NAME = "eliceai/korean-fineweb-edu-demo"
SAMPLE_COUNT = 5  # 튜터링을 위해 샘플 5개만 분석해 볼게요!

def load_dataset_with_fallback(dataset_id: str, split: str, streaming: bool) -> object:
    """
    스트리밍 모드를 시도하고, 실패할 경우 일반 로딩 모드로 대체하는 안전한 데이터셋 로더.
    (AI 튜터가 가장 좋아하는 '예외 처리' 패턴이에요! 🛡️)
    """
    print("=" * 80)
    print(f"✨ [Step 1/3] 데이터 로드 시작: {dataset_id} ({split} 스플릿)")
    
    # 1. 스트리밍 모드 시도 (가장 빠르고 메모리 효율적!)
    try:
        dataset = load_dataset(dataset_id, split=split, streaming=True)
        print("✅ 성공! 스트리밍 모드로 데이터셋을 로드했어요. 메모리 걱정 끝!")
        return dataset
    except Exception as e:
        print(f"⚠️ 스트리밍 로드에 실패했어요. 오류: {e}")
        print("🔄 일반(Non-streaming) 모드로 재시도합니다...")
        
        # 2. 일반 로딩 모드로 대체
        try:
            dataset = load_dataset(dataset_id, split=split, streaming=False)
            print("✅ 성공! 일반 모드로 데이터셋을 로드했어요. 이제 전체 데이터셋에 접근할 수 있어요.")
            return dataset
        except Exception as e_fall:
            print(f"💔 치명적인 오류: 데이터셋 로드에 실패했습니다. ({e_fall})")
            return None

def extract_educational_info(text: str) -> dict:
    """
    💡 창의적인 실습 부분! 이 함수는 raw 텍스트에서 핵심 지식 구조를 '가상으로' 추출합니다.
    (실제로는 복잡한 NLP 모델이 필요하지만, 여기서는 간단한 로직으로 시뮬레이션!)
    """
    
    # 1. 주제 추출 (가장 처음 나오는 핵심 키워드를 주제로 설정)
    # 간단하게 텍스트의 첫 문장을 사용합니다.
    topic = text.strip()
    if len(topic) > 100:
        topic = topic[:100] + "..."
        
    # 2. '정의' 및 '예시' 구조 추출 시뮬레이션
    
    # 텍스트에서 '은/는 ~이다', '의미는 ~이다' 같은 교육적 패턴을 찾기 위해 간단한 정규식 사용
    definition_match = re.search(r'(.*?)(은|는|이다|입니다)[^\.]*?[\.\?]', text)
    definition = "🔎 정의를 추출하기 어려웠어요. (텍스트가 너무 자유로운 에세이일 수 있어요.)"
    
    if definition_match:
        # 정의의 시작 부분을 대략적으로 추출 (과도한 복잡성을 피하기 위해 간단히 처리)
        start_index = definition_match.group(1)
        # 뒤로 10글자만 잘라 예시로 사용합니다.
        definition = f"📄 [정의 추정]: {start_index[-30:]}..."
    
    # 3. 핵심 내용 요약 (가장 긴 문장이나 문단의 시작 부분)
    # 텍스트를 문장 단위로 분리하여, 가장 긴 첫 문장을 요약 예시로 사용합니다.
    sentences = re.split(r'[?.!]\s*', text)
    summary = "📝 [핵심 요약]: 텍스트 전체에서 가장 중요한 내용을 LLM이 뽑아낼 수 있도록, 첫 문장의 분위기만 가져와 봤어요."
    
    if sentences and sentences[0]:
        # 첫 문장으로 요약을 시뮬레이션
        summary = f"📝 [핵심 요약]: {sentences[0].strip()}"
        
    return {
        "Topic": topic,
        "Definition": definition,
        "Summary": summary
    }

def main_tutor_script():
    """
    ✨ 초급 AI 실습: 교육용 텍스트에서 지식 구조화하기 (Concept Extraction Simulation)
    """
    print("\n" + "🎉" * 80)
    print("✨ AI 튜터와 함께하는 초급 데이터 구조화 실습에 오신 것을 환영합니다! ✨")
    print("🧐 목표: 거대한 텍스트 덩어리에서 '정의', '예시', '핵심 개념'을 분리하는 시뮬레이션을 해볼 거예요.")
    print("💡 이 작업은 '정보 추출(Information Extraction)'이라는 멋진 AI 분야에 속해요!")
    print("=" * 80)

    # 1. 데이터 로드 (제약 조건 준수: 스트리밍 및 Fallback)
    dataset_iterator = load_dataset_with_fallback(
        dataset_id=DATASET_NAME, 
        split='train', 
        streaming=True
    )
    
    if dataset_iterator is None:
        print("\n🚨 데이터를 로드할 수 없어 스크립트를 종료합니다. 라이브러리나 네트워크를 확인해 주세요!")
        return

    # 2. 샘플링 및 반복 처리 (제약 조건 준수: take() 패턴 사용)
    print("\n" + "~" * 80)
    print(f"🚀 [Step 2/3] 데이터 샘플 분석 ({SAMPLE_COUNT}개 샘플)")
    print("튜터: 이제 메모리 부하 없이, 첫 {SAMPLE_COUNT}개의 샘플만 뽑아서 분석을 시작할게요!")
    print("~" * 80)

    # take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    if hasattr(dataset_iterator, "take"):
        sample_iterator = iter(dataset_iterator.take(SAMPLE_COUNT))
    else:
        # 일반 데이터셋 (Dataset)
        sample_iterator = iter(list(dataset_iterator)[:SAMPLE_COUNT])

    # 3. 실습 실행 루프
    results = []
    for i, sample_data in enumerate(tqdm(sample_iterator, desc="🧠 분석 중")):
        # 데이터를 추출하고 공백 처리를 해줍니다.
        raw_text = sample_data['text'].strip()
        
        if not raw_text:
            print(f"  [{i+1}번째 샘플]: 비어 있는 텍스트라서 건너뜁니다.")
            continue
            
        # 핵심 함수 호출
        extracted_info = extract_educational_info(raw_text)
        results.append(extracted_info)

    # 4. 결과 출력 및 분석
    print("\n" + "=" * 80)
    print(f"🎉 [Step 3/3] 분석 완료! {len(results)}개의 구조화된 개념을 추출했어요.")
    print("축하해요! 😊 raw 텍스트에서 구조화된 지식을 뽑아내는 과정, 바로 이게 LLM의 핵심 능력 중 하나랍니다!")
    print("=" * 80)
    
    for i, result in enumerate(results):
        print(f"\n🌟 [샘플 {i+1}의 구조화 결과]")
        print("-" * 30)
        print(f"📝 원문 기반 핵심 주제 (Topic): {result['Topic']}")
        print(f"✨ 정의 (Definition): {result['Definition']}")
        print(f"🎯 요약 예시 (Summary): {result['Summary']}")
        
    print("\n\n✨ ✨ ✨ 튜터 코멘트 ✨ ✨ ✨")
    print("🔍 우리가 한 작업: raw 텍스트(예: '어떤 웹 페이지의 글')를 받아서, '주제', '정의', '핵심 요약'이라는 규칙적인 JSON/Dictionary 형태의 데이터로 변환했습니다.")
    print("📚 이 과정의 이름: '정보 추출 (Information Extraction)' 또는 '요약 (Summarization)'입니다.")
    print("💻 AI 튜터의 꿀팁: 실제 상업용 서비스에서는 이 단계에서 나이팅게일(NLTK), Spacy, 또는 트랜스포머 기반 모델(BERT, GPT 등)을 사용해 훨씬 정확하게 문법적 역할을 분석해요!")
    print("🚀 다음 목표는? 이 구조화된 지식을 바탕으로 '퀴즈 문제'를 자동으로 생성해보는 거예요! 정말 재미있을 거예요. 다음 시간도 기대해주세요! 👋")

if __name__ == "__main__":
    main_tutor_script()